In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("../data.csv", encoding="cp1252")
df.head()

C:\Users\Riddhi Kale\AppData\Local\Temp\ipykernel_13908\3853325627.py:1: DtypeWarning: Columns (0: stn_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data.csv", encoding="cp1252")


,stn_code,sampling_date,state,location,agency,type,so2,no2,rspm,spm,location_monitoring_station,pm2_5,date
0,150.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",4.8,17.4,NaN,NaN,NaN,NaN,1990-02-01
1,151.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,Industrial Area,3.1,7.0,NaN,NaN,NaN,NaN,1990-02-01
2,152.0,February - M021990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",6.2,28.5,NaN,NaN,NaN,NaN,1990-02-01
3,150.0,March - M031990,Andhra Pradesh,Hyderabad,NaN,"Residential, Rural and other Areas",6.3,14.7,NaN,NaN,NaN,NaN,1990-03-01
4,151.0,March - M031990,Andhra Pradesh,Hyderabad,NaN,Industrial Area,4.7,7.5,NaN,NaN,NaN,NaN,1990-03-01


In [7]:
nulls = df.isnull().sum()
nulls

stn_code                       144077
sampling_date                       3
state                               0
location                            3
agency                         149481
type                             5393
so2                             34646
no2                             16233
rspm                            40222
spm                            237387
location_monitoring_station     27491
pm2_5                          426428
date                                7
dtype: int64

In [9]:
null_pct = (nulls / len(df) * 100).round(2)
null_pct

stn_code                       33.06
sampling_date                   0.00
state                           0.00
location                        0.00
agency                         34.30
type                            1.24
so2                             7.95
no2                             3.73
rspm                            9.23
spm                            54.48
location_monitoring_station     6.31
pm2_5                          97.86
date                            0.00
dtype: float64

In [10]:
print(pd.DataFrame({"missing": nulls, "percent": null_pct}))

                             missing  percent
stn_code                      144077    33.06
sampling_date                      3     0.00
state                              0     0.00
location                           3     0.00
agency                        149481    34.30
type                            5393     1.24
so2                            34646     7.95
no2                            16233     3.73
rspm                           40222     9.23
spm                           237387    54.48
location_monitoring_station    27491     6.31
pm2_5                         426428    97.86
date                               7     0.00


### Null pattern finding

pm2_5 is missing 97.86% of the time across India.
This means most Indian cities simply did not have PM2.5 sensors when this data was collected. It is not a mistake in the data — it is a gap in monitoring infrastructure.

Delhi is one of the very few cities with PM2.5 readings, which is why we focused on Delhi all week.

spm is missing 54% — it is an older pollution measure that was being replaced by pm2_5 over time.

so2 and rspm are only missing 7–9% — much more usable.

For the Health Risk Scorer: I cannot use pm2_5 as a feature for all of India — there is not enough data.
rspm (PM10) is the better choice for a national model.

In [11]:
so2_nulls = df.groupby("location")["so2"].apply(
    lambda x: x.isnull().sum()
).sort_values(ascending=False)